In [ ]:
import sys
import torch
from pathlib import Path
from IPython.display import display, HTML

# import utils from parent directory
sys.path.append(str(Path.cwd().parent))

from utils import visualize_saliency
from model.hf_extractor import HFExtractor
from sae.models.jumprelu_sae import JumpReLUSAE

HF_MODEL_PATH = "/home/liamdugan_google_com/detector-scope/trained_models/20250929_191747_LR1e-06"
SAE_MODEL_PATH = "/home/liamdugan_google_com/detector-scope/tensorboard/detectgemma_indep_search/num_epochs/run_20251015_155058_bs128_bw0.001_lr7e-05_sp0.0006_ne30_yes/finetuned_sae.pth"

# Load HF model and SAE model
extractor = HFExtractor(HF_MODEL_PATH, default_hooks=["norm"])
state_dict = torch.load(SAE_MODEL_PATH)
sae_model = JumpReLUSAE(
    d_input=state_dict["W_enc"].shape[0], 
    d_hidden=state_dict["W_enc"].shape[1]
)
sae_model.load_state_dict(state_dict)
sae_model = sae_model.to(extractor.device).eval()

In [ ]:
text = 'This review would decrease the chances of you buying the product, as the reviewer highlights several negative aspects of the product and criticizes its quality.'

In [ ]:
tokens, _, saliency = extractor.get_gradient_based_saliency(text, sae_model)
for (feature_id, grad_tensor) in saliency.items():
    html = visualize_saliency(tokens, grad_tensor, feature_id)
    display(HTML(html))